In [1]:
from pathlib import Path

base = "C:/Users/colin/projects/UW/Project/LA/LA"
train_files = base + "/ASVspoof2019_LA_train/flac"
dev_files = base + "/ASVspoof2019_LA_dev/flac"
eval_files = base + "/ASVspoof2019_LA_eval/flac"
train_protocols = base + "/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"
dev_protocols = base + "/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt"
eval_protocols = base + "/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt"

In [2]:
train_audio_files = list(Path(train_files).glob("*.flac"))
dev_audio_files = list(Path(dev_files).glob("*.flac"))
eval_audio_files = list(Path(eval_files).glob("*.flac"))

In [3]:
import pandas as pd

train_df = pd.read_csv(train_protocols, sep=r"\s+", header=None)
dev_df = pd.read_csv(dev_protocols, sep=r"\s+", header=None)
eval_df = pd.read_csv(eval_protocols, sep=r"\s+", header=None)

In [4]:
label_map = {
    **dict(zip(train_df[1], train_df[4])),
    **dict(zip(dev_df[1], dev_df[4])),
    **dict(zip(eval_df[1], eval_df[4]))
}

In [5]:
import torch.nn as nn
import torch.nn.functional as F

SR = 16000
N_MELS = 80
N_FFT = 1024
HOP_LENGTH = 160
WIN_LENGTH = 400
FMIN = 20
FMAX = 7600
MAX_SECONDS = 4.0
MAX_FRAMES = 400

class BiggerSpecCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.1)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.15)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2)
        )

        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout2d(0.25)
        )

        self.fc1 = nn.Linear(256, 128)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        x = x.mean(dim=(2, 3))

        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        logits = self.fc2(x).squeeze(1)

        return logits

In [6]:
import librosa
import torch
import numpy as np
import random

def specaugment(logmel):
    n_mels, n_frames = logmel.shape

    # Frequency mask
    if random.random() < 0.5:
        max_f = int(n_mels * 0.2)
        f = random.randint(1, max_f)
        f0 = random.randint(0, n_mels - f)
        logmel[f0:f0+f, :] = logmel.min()

    # Time mask
    if random.random() < 0.5:
        max_t = int(n_frames * 0.2)
        t = random.randint(1, max_t)
        t0 = random.randint(0, n_frames - t)
        logmel[:, t0:t0+t] = logmel.min()

    return logmel

def compute_logmel(audio_path, with_masking=False):
    y, sr = librosa.load(str(audio_path), sr=SR, mono=True)

    target_len = int(MAX_SECONDS * SR)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]

    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        n_mels=N_MELS,
        fmin=FMIN,
        fmax=FMAX,
        power=2.0
    )

    logmel = librosa.power_to_db(mel, ref=np.max)

    T = logmel.shape[1]
    if T < MAX_FRAMES:
        pad_val = logmel.min()
        logmel = np.pad(logmel, ((0, 0), (0, MAX_FRAMES - T)), constant_values=pad_val)
    else:
        logmel = logmel[:, :MAX_FRAMES]

    if with_masking:
        logmel = specaugment(logmel)

    return torch.from_numpy(logmel).unsqueeze(0).float()

In [7]:
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

class AudioDataset(Dataset):
    def __init__(self, audio_files, with_masking=False):
        self.audio_files = []
        self.labels = []
        self.with_masking = with_masking

        for audio_path in audio_files:
            audio_id = audio_path.stem
            if audio_id not in label_map:
                continue
            self.audio_files.append(audio_path)
            label = 0 if label_map[audio_id] == "bonafide" else 1
            self.labels.append(label)

    def __len__(self):
        return len(self.audio_files)

    def __getitem__(self, idx):
        path = self.audio_files[idx]
        label = self.labels[idx]
        x = compute_logmel(path, with_masking=self.with_masking)
        return x, torch.tensor(label).float()

def establish_dataset(audio_files, with_masking=False):
    return AudioDataset(audio_files, with_masking=with_masking)

In [8]:
train_dataset = establish_dataset(train_audio_files)
dev_dataset = establish_dataset(dev_audio_files)
eval_dataset = establish_dataset(eval_audio_files)

In [9]:
# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False)
eval_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model
model = BiggerSpecCNN().to(device)

# Other
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Training
num_epochs = 10
for epoch in range(num_epochs):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch_X, batch_y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        outputs = model(batch_X)

        loss = criterion(outputs, batch_y)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        preds = torch.sigmoid(outputs) > 0.5

        correct += (preds == batch_y.bool()).sum().item()
        total += batch_y.size(0)

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Epoch loss: {epoch_loss:.4f} "
          f"Epoch accuracy: {epoch_acc:.4f}")

model.eval()
print("Training complete.")

Epoch 1/10: 100%|████████████████████████████████████████████████████████████████████| 794/794 [43:06<00:00,  3.26s/it]


Epoch [1/10] Epoch loss: 0.2746 Epoch accuracy: 0.8978


Epoch 2/10: 100%|████████████████████████████████████████████████████████████████████| 794/794 [21:28<00:00,  1.62s/it]


Epoch [2/10] Epoch loss: 0.2171 Epoch accuracy: 0.8984


Epoch 3/10: 100%|████████████████████████████████████████████████████████████████████| 794/794 [18:42<00:00,  1.41s/it]


...

Epoch [8/10] Epoch loss: 0.0825 Epoch accuracy: 0.9699


Epoch 9/10: 100%|████████████████████████████████████████████████████████████████████| 794/794 [18:57<00:00,  1.43s/it]


Epoch [9/10] Epoch loss: 0.0675 Epoch accuracy: 0.9754


Epoch 10/10: 100%|███████████████████████████████████████████████████████████████████| 794/794 [18:26<00:00,  1.39s/it]

Epoch [10/10] Epoch loss: 0.0566 Epoch accuracy: 0.9799
Training complete.


In [10]:
torch.save(model.state_dict(), "./cnn_weights/baseline7")

In [11]:
y_dev_pred = []
y_dev_scores = []
with torch.no_grad():
    for batch_X, _ in tqdm(dev_loader, desc="Dev inference"):
        batch_X = batch_X.to(device)
        outputs = model(batch_X)
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).cpu().numpy()
        scores = probs.cpu().numpy()
        y_dev_pred.extend(preds)
        y_dev_scores.extend(scores)
y_dev_pred = np.array(y_dev_pred)
y_dev_scores = np.array(y_dev_scores)

y_eval_pred = []
y_eval_scores = []
with torch.no_grad():
    for batch_X, _ in tqdm(eval_loader, desc="Eval inference"):
        batch_X = batch_X.to(device)
        outputs = model(batch_X)
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).cpu().numpy()
        scores = probs.cpu().numpy()
        y_eval_pred.extend(preds)
        y_eval_scores.extend(scores)
y_eval_pred = np.array(y_eval_pred)
y_eval_scores = np.array(y_eval_scores)

Eval inference: 100%|██████████████████████████████████████████████████████████████| 2227/2227 [33:41<00:00,  1.10it/s]


In [12]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_dev = np.array(dev_dataset.labels)
accuracy = accuracy_score(y_dev, y_dev_pred)
print(f"Model Accuracy (against dev): {accuracy:.2f}")
confusion = confusion_matrix(y_dev, y_dev_pred)
print(confusion)

y_eval = np.array(eval_dataset.labels)
accuracy = accuracy_score(y_eval, y_eval_pred)
print(f"Model Accuracy (against eval): {accuracy:.2f}")
confusion = confusion_matrix(y_eval, y_eval_pred)
print(confusion)

print("\n----------\n")

classification = classification_report(y_dev, y_dev_pred)
print("Classification Report (against dev):")
print(classification)
classification = classification_report(y_eval, y_eval_pred)
print("Classification Report (against eval):")
print(classification)

Model Accuracy (against dev): 0.96
[[ 1604   944]
 [   27 22269]]
Model Accuracy (against eval): 0.93
[[ 5516  1839]
 [ 3296 60586]]

----------

Classification Report (against dev):
              precision    recall  f1-score   support

           0       0.98      0.63      0.77      2548
           1       0.96      1.00      0.98     22296

    accuracy                           0.96     24844
   macro avg       0.97      0.81      0.87     24844
weighted avg       0.96      0.96      0.96     24844

Classification Report (against eval):
              precision    recall  f1-score   support

           0       0.63      0.75      0.68      7355
           1       0.97      0.95      0.96     63882

    accuracy                           0.93     71237
   macro avg       0.80      0.85      0.82     71237
weighted avg       0.93      0.93      0.93     71237



In [13]:
from sklearn.metrics import roc_curve, roc_auc_score

def compute_eer(y_true, y_scores):
    fpr, tpr, thresholds = roc_curve(y_true, y_scores, pos_label=1)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    eer = (fpr[idx] + fnr[idx]) / 2.0
    return eer

auc_score = roc_auc_score(y_dev, y_dev_scores)
print(f"Model AUC (against dev): {auc_score:.2f}")
err_score = compute_eer(y_dev, y_dev_scores)
print(f"Model ERR (against dev): {err_score:.2f}")

auc_score = roc_auc_score(y_eval, y_eval_scores)
print(f"Model AUC (against eval): {auc_score:.2f}")
err_score = compute_eer(y_eval, y_eval_scores)
print(f"Model ERR (against eval): {err_score:.2f}")

Model AUC (against dev): 0.99
Model ERR (against dev): 0.03
Model AUC (against eval): 0.96
Model ERR (against eval): 0.10


In [14]:
# import torch

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = BiggerSpecCNN().to(device)
# model.load_state_dict(torch.load("./cnn_weights/baseline7"))
# model.eval()